<a href="https://colab.research.google.com/github/DanishShah619/git_agent/blob/main/git_man.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
!git clone --depth 1 https://github.com/git/git.git


Cloning into 'git'...
remote: Enumerating objects: 4933, done.
remote: Counting objects: 100% (4933/4933), done.
remote: Compressing objects: 100% (4323/4323), done.
remote: Total 4933 (delta 498), reused 2165 (delta 439), pack-reused 0 (from 0)
Receiving objects: 100% (4933/4933), 12.35 MiB | 7.60 MiB/s, done.
Resolving deltas: 100% (498/498), done.


In [6]:
import os
doc_files = [f for f in os.listdir('git/Documentation') if f.startswith('git-') and f.endswith('.adoc')]
print(len(doc_files), "command doc files found")
print(doc_files[:10])

169 command doc files found
['git-remote-fd.adoc', 'git-last-modified.adoc', 'git-submodule.adoc', 'git-fast-import.adoc', 'git-backfill.adoc', 'git-prune.adoc', 'git-cvsimport.adoc', 'git-refs.adoc', 'git-instaweb.adoc', 'git-http-fetch.adoc']


In [7]:
with open('git/Documentation/git-rebase.adoc', 'r') as f:
    content = f.read()

print(content[:2000])

git-rebase(1)

NAME
----
git-rebase - Reapply commits on top of another base tip

SYNOPSIS
--------
[verse]
'git rebase' [-i | --interactive] [<options>] [--exec <cmd>]
	[--onto <newbase> | --keep-base] [<upstream> [<branch>]]
'git rebase' [-i | --interactive] [<options>] [--exec <cmd>] [--onto <newbase>]
	--root [<branch>]
'git rebase' (--continue|--skip|--abort|--quit|--edit-todo|--show-current-patch)

DESCRIPTION
-----------
Transplant a series of commits onto a different starting point.
You can also use `git rebase` to reorder or combine commits: see INTERACTIVE
MODE below for how to do that.

For example, imagine that you have been working on the `topic` branch in this
history, and you want to "catch up" to the work done on the `master` branch.

------------
          A---B---C topic
         /
    D---E---F---G master
------------

You want to transplant the commits you made on `topic` since it diverged from
`master` (i.e. A, B, and C), on top of the current `master`.  You can do

In [8]:
import re

def parse_adoc_file(filepath):
    with open(filepath, 'r', errors='ignore') as f:
        text = f.read()

    cmd_name = os.path.basename(filepath).replace('.adoc', '')

    # Match: an all-caps header line, followed by a line of dashes
    pattern = re.compile(r'^([A-Z][A-Z \-]{2,})\n-{3,}\n', re.MULTILINE)

    matches = list(pattern.finditer(text))
    chunks = []

    for i, m in enumerate(matches):
        header = m.group(1).strip()
        start = m.end()
        end = matches[i+1].start() if i + 1 < len(matches) else len(text)
        body = text[start:end].strip()
        if body:
            chunks.append({
                "text": f"{header}\n{body}",
                "metadata": {"command": cmd_name, "section": header}
            })

    return chunks

In [9]:
chunks = parse_adoc_file('git/Documentation/git-rebase.adoc')
print(len(chunks), "chunks found")
for c in chunks:
    print("---", c['metadata']['section'], "---")
    print(c['text'][:150])
    print()

15 chunks found
--- NAME ---
NAME
git-rebase - Reapply commits on top of another base tip

--- SYNOPSIS ---
SYNOPSIS
[verse]
'git rebase' [-i | --interactive] [<options>] [--exec <cmd>]
	[--onto <newbase> | --keep-base] [<upstream> [<branch>]]
'git rebase' [

--- DESCRIPTION ---
DESCRIPTION
Transplant a series of commits onto a different starting point.
You can also use `git rebase` to reorder or combine commits: see INTERACTI

--- TRANSPLANTING A TOPIC BRANCH WITH --ONTO ---
TRANSPLANTING A TOPIC BRANCH WITH --ONTO
Here is how you would transplant a topic branch based on one
branch to another, to pretend that you forked th

--- MODE OPTIONS ---
MODE OPTIONS
The options in this section cannot be used with any other option,
including not with each other:

--continue::
	Restart the rebasing proc

--- OPTIONS ---
OPTIONS
--onto <newbase>::
	Starting point at which to create the new commits. If the
	`--onto` option is not specified, the starting point is
	`<upst

--- INCOMPATIBLE OPTIONS -

In [10]:
SKIP_SECTIONS = {"GIT"}  # boilerplate footer, no useful content

def parse_adoc_file(filepath):
    with open(filepath, 'r', errors='ignore') as f:
        text = f.read()

    cmd_name = os.path.basename(filepath).replace('.adoc', '')

    # Match: an all-caps header line, followed by a line of dashes
    pattern = re.compile(r'^([A-Z][A-Z \-]{2,})\n-{3,}\n', re.MULTILINE)

    matches = list(pattern.finditer(text))
    chunks = []

    for i, m in enumerate(matches):
        header = m.group(1).strip()
        if header in SKIP_SECTIONS:
            continue
        start = m.end()
        end = matches[i+1].start() if i + 1 < len(matches) else len(text)
        body = text[start:end].strip()
        if body:
            chunks.append({
                "text": f"{header}\n{body}",
                "metadata": {"command": cmd_name, "section": header}
            })

    return chunks

In [11]:
all_chunks = []
for f in doc_files:
    all_chunks.extend(parse_adoc_file(os.path.join('git/Documentation', f)))

print(len(all_chunks), "total chunks across all commands")

# sanity check: distribution of section types
from collections import Counter
section_counts = Counter(c['metadata']['section'] for c in all_chunks)
print(section_counts.most_common(15))

1135 total chunks across all commands
[('NAME', 167), ('SYNOPSIS', 167), ('DESCRIPTION', 167), ('OPTIONS', 145), ('EXAMPLES', 72), ('SEE ALSO', 72), ('CONFIGURATION', 50), ('OUTPUT', 17), ('DISCUSSION', 16), ('COMMANDS', 15), ('NOTES', 10), ('BUGS', 10), ('CAVEATS', 9), ('FILES', 8), ('ENVIRONMENT', 6)]


In [12]:
!pip install -q sentence-transformers chromadb

In [13]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('all-MiniLM-L6-v2')

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [14]:
import chromadb

client = chromadb.Client()
collection = client.get_or_create_collection(name="git_docs")

texts = [c["text"] for c in all_chunks]
metadatas = [c["metadata"] for c in all_chunks]
ids = [f"{c['metadata']['command']}_{c['metadata']['section']}_{i}" for i, c in enumerate(all_chunks)]

embeddings = model.encode(texts, show_progress_bar=True, batch_size=64)

collection.add(
    embeddings=embeddings.tolist(),
    documents=texts,
    metadatas=metadatas,
    ids=ids
)

print(collection.count(), "chunks added to Chroma")

Batches:   0%|          | 0/18 [00:00<?, ?it/s]

1135 chunks added to Chroma


In [15]:
import chromadb
import os

# Define a persistent directory for ChromaDB
CHROMA_PERSIST_PATH = './chroma_db_git_docs'

# Initialize a persistent client
# If the directory exists, it will load the existing database
# If not, it will create a new one
client = chromadb.PersistentClient(path=CHROMA_PERSIST_PATH)

# Try to get the collection, create it if it doesn't exist
try:
    collection = client.get_collection(name="git_docs")
    print(f"Loaded existing collection '{collection.name}' with {collection.count()} items.")
except:
    collection = client.create_collection(name="git_docs")
    print(f"Created new collection '{collection.name}'.")

# Now, re-add your chunks to the (potentially new or empty) collection.
# You might want to add logic here to only add new chunks or update existing ones.
# For simplicity, we'll clear and re-add for demonstration, but in a real pipeline
# you'd implement incremental updates.

# If the collection is empty, populate it
if collection.count() == 0:
    print("Collection is empty, populating with all_chunks...")
    texts = [c["text"] for c in all_chunks]
    metadatas = [c["metadata"] for c in all_chunks]
    ids = [f"{c['metadata']['command']}_{c['metadata']['section']}_{i}" for i, c in enumerate(all_chunks)]

    # Make sure 'model' (SentenceTransformer) is loaded before encoding
    # This assumes 'model' is defined in an earlier cell and available in the kernel
    if 'model' not in locals():
        from sentence_transformers import SentenceTransformer
        model = SentenceTransformer('all-MiniLM-L6-v2')

    embeddings = model.encode(texts, show_progress_bar=True, batch_size=64)

    collection.add(
        embeddings=embeddings.tolist(),
        documents=texts,
        metadatas=metadatas,
        ids=ids
    )
    print(f"Populated collection with {collection.count()} chunks.")
else:
    print(f"Collection already contains {collection.count()} chunks. Skipping re-population.")



Created new collection 'git_docs'.
Collection is empty, populating with all_chunks...


Batches:   0%|          | 0/18 [00:00<?, ?it/s]

Populated collection with 1135 chunks.


In [16]:
query = "how do I undo my last commit"
query_embedding = model.encode([query])

results = collection.query(
    query_embeddings=query_embedding.tolist(),
    n_results=5
)

for doc, meta, dist in zip(results['documents'][0], results['metadatas'][0], results['distances'][0]):
    if meta.get('source') == 'progit':
        print(f"[{meta.get('source')} / {meta.get('chapter')} / {meta.get('heading')}] (dist={dist:.3f})")
    else:
        print(f"[{meta.get('command')} / {meta.get('section')}] (dist={dist:.3f})")
    print(doc[:200])
    print()

[git-revert / SYNOPSIS] (dist=0.769)
SYNOPSIS
[verse]
'git revert' [--[no-]edit] [-n] [-m <parent-number>] [-s] [-S[<keyid>]] <commit>...
'git revert' (--continue | --skip | --abort | --quit)

[git-revert / NAME] (dist=0.818)
NAME
git-revert - Revert some existing commits

[git-revert / DESCRIPTION] (dist=0.879)
DESCRIPTION
Given one or more existing commits, revert the changes that the
related patches introduce, and record some new commits that record
them.  This requires your working tree to be clean (no mo

[git-reset / DESCRIPTION] (dist=0.906)
DESCRIPTION
`git reset` does either of the following:

1. `git reset [<mode>] <commit>` changes which commit `HEAD` points to. This
   makes it possible to undo various Git operations, for example com

[git-revert / OPTIONS] (dist=0.944)
OPTIONS
<commit>...::
	Commits to revert.
	For a more complete list of ways to spell commit names, see
	linkgit:gitrevisions[7].
	Sets of commits can also be given but no traversal is done by
	default



In [17]:
!git clone --depth 1 https://github.com/progit/progit2.git

Cloning into 'progit2'...
remote: Enumerating objects: 505, done.
remote: Counting objects: 100% (505/505), done.
remote: Compressing objects: 100% (368/368), done.
remote: Total 505 (delta 125), reused 378 (delta 124), pack-reused 0 (from 0)
Receiving objects: 100% (505/505), 14.05 MiB | 18.16 MiB/s, done.
Resolving deltas: 100% (125/125), done.


In [18]:
import os
book_dir = 'progit2/book'
for root, dirs, files in os.walk(book_dir):
    print(root, len(files), "files")

progit2/book 9 files
progit2/book/B-embedding-git 0 files
progit2/book/B-embedding-git/callouts 20 files
progit2/book/B-embedding-git/sections 5 files
progit2/book/09-git-and-other-scms 0 files
progit2/book/09-git-and-other-scms/sections 7 files
progit2/book/A-git-in-other-environments 0 files
progit2/book/A-git-in-other-environments/sections 8 files
progit2/book/03-git-branching 0 files
progit2/book/03-git-branching/sections 6 files
progit2/book/10-git-internals 0 files
progit2/book/10-git-internals/sections 8 files
progit2/book/02-git-basics 0 files
progit2/book/02-git-basics/sections 7 files
progit2/book/07-git-tools 1 files
progit2/book/07-git-tools/callouts 20 files
progit2/book/07-git-tools/sections 15 files
progit2/book/06-github 0 files
progit2/book/06-github/callouts 20 files
progit2/book/06-github/sections 5 files
progit2/book/04-git-server 0 files
progit2/book/04-git-server/sections 9 files
progit2/book/05-distributed-git 0 files
progit2/book/05-distributed-git/sections 3 fi

In [19]:
import re

def parse_progit_file(filepath, book_root='progit2/book'):
    with open(filepath, 'r', errors='ignore') as f:
        text = f.read()

    # Clean noise
    text = re.sub(r'\(\(.*?\)\)', '', text)              # index terms
    text = re.sub(r'\[\[.*?\]\]', '', text)                   # anchors
    text = re.sub(r'image::.*?\[.*?\]', '', text)             # image refs
    text = re.sub(r'^\.[A-Z].*$', '', text, flags=re.MULTILINE)  # caption lines


    rel_path = os.path.relpath(filepath, book_root)
    parts = rel_path.split(os.sep)
    chapter = parts[0] if len(parts) > 0 else "unknown"
    section_file = os.path.splitext(parts[-1])[0]


    pattern = re.compile(r'^(={2,5})\s+(.+)$', re.MULTILINE)
    matches = list(pattern.finditer(text))

    chunks = []
    if not matches:

        body = text.strip()
        if body:
            chunks.append({
                "text": body,
                "metadata": {"chapter": chapter, "section_file": section_file, "heading": section_file, "source": "progit"}
            })
        return chunks

    for i, m in enumerate(matches):
        heading = m.group(2).strip()
        start = m.end()
        end = matches[i+1].start() if i + 1 < len(matches) else len(text)
        body = text[start:end].strip()
        if body:
            chunks.append({
                "text": f"{heading}\n{body}",
                "metadata": {"chapter": chapter, "section_file": section_file, "heading": heading, "source": "progit"}
            })

    return chunks

In [20]:
all_progit_chunks = []

for root, dirs, files in os.walk(book_dir):
    for file in files:
        if file.endswith('.asc'):
            filepath = os.path.join(root, file)
            chunks = parse_progit_file(filepath, book_root=book_dir)
            all_progit_chunks.extend(chunks)

print(f"Total initial chunks: {len(all_progit_chunks)}")

Total initial chunks: 533


In [21]:
!pip install -q sentence-transformers

In [22]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('all-MiniLM-L6-v2')

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [23]:
# The 'client' object (chromadb.PersistentClient) is already initialized in cell 6fc11324.
# The 'model' object (SentenceTransformer) is also initialized in cell FzsBvoqh5Kgo.

# Get the existing persistent collection. It should now contain chunks from both 'git' and 'progit2' if this cell is run.
collection = client.get_or_create_collection(name="git_docs")

texts = [c["text"] for c in all_progit_chunks]
metadatas = [c["metadata"] for c in all_progit_chunks]
# Ensure IDs are unique across both datasets. Using 'chapter_section_file_index' for progit2 should prevent collisions.
ids = [f"progit_{c['metadata']['chapter']}_{c['metadata']['section_file']}_{i}" for i, c in enumerate(all_progit_chunks)]

print(f"Encoding {len(texts)} chunks from progit2 documentation...")
embeddings = model.encode(texts, show_progress_bar=True, batch_size=64)

# Add the new chunks to the persistent collection
collection.add(
    embeddings=embeddings.tolist(),
    documents=texts,
    metadatas=metadatas,
    ids=ids
)

print(f"Added {len(texts)} chunks from progit2 to Chroma. Total chunks in collection: {collection.count()}")

Encoding 533 chunks from progit2 documentation...


Batches:   0%|          | 0/9 [00:00<?, ?it/s]

Added 533 chunks from progit2 to Chroma. Total chunks in collection: 1668


In [24]:
import requests
import time

def fetch_git_questions(pages=5, pagesize=100):
    all_questions = []
    for page in range(1, pages + 1):
        response = requests.get(
            "https://api.stackexchange.com/2.3/questions",
            params={
                "page": page,
                "pagesize": pagesize,
                "order": "desc",
                "sort": "votes",
                "tagged": "git",
                "site": "stackoverflow",
                "filter": "withbody"
            }
        )
        data = response.json()
        all_questions.extend(data.get("items", []))
        if not data.get("has_more", False):
            break
        time.sleep(1)  # be polite to the API, avoid rate limiting
    return all_questions

questions = fetch_git_questions(pages=5)
print(len(questions), "questions fetched")
print(questions[0].keys())

500 questions fetched
dict_keys(['tags', 'owner', 'is_answered', 'view_count', 'protected_date', 'accepted_answer_id', 'answer_count', 'community_owned_date', 'score', 'last_activity_date', 'creation_date', 'last_edit_date', 'question_id', 'content_license', 'link', 'title', 'body'])


In [25]:
def fetch_answers(question_ids, batch_size=30):
    all_answers = {}
    for i in range(0, len(question_ids), batch_size):
        batch = question_ids[i:i+batch_size]
        ids_str = ";".join(str(qid) for qid in batch)
        response = requests.get(
            f"https://api.stackexchange.com/2.3/questions/{ids_str}/answers",
            params={
                "order": "desc",
                "sort": "votes",
                "site": "stackoverflow",
                "filter": "withbody"
            }
        )
        data = response.json()
        for a in data.get("items", []):
            qid = a["question_id"]
            # keep only the top-voted answer per question
            if qid not in all_answers or a["score"] > all_answers[qid]["score"]:
                all_answers[qid] = a
        time.sleep(1)
    return all_answers

question_ids = [q["question_id"] for q in questions]
answers = fetch_answers(question_ids)
print(len(answers), "answers fetched")

466 answers fetched


In [26]:
from bs4 import BeautifulSoup

def clean_html(raw_html):
    soup = BeautifulSoup(raw_html, "html.parser")
    return soup.get_text(separator="\n").strip()

so_chunks = []
for q in questions:
    qid = q["question_id"]
    if qid not in answers:
        continue  # skip questions with no fetched answer

    a = answers[qid]
    question_text = clean_html(q["body"])
    answer_text = clean_html(a["body"])

    combined_text = f"Question: {q['title']}\n{question_text}\n\nAnswer:\n{answer_text}"

    so_chunks.append({
        "text": combined_text,
        "metadata": {
            "source": "stackoverflow",
            "question_id": qid,
            "title": q["title"],
            "question_score": q["score"],
            "answer_score": a["score"],
            "tags": ",".join(q.get("tags", [])),
            "link": q["link"]
        }
    })

print(len(so_chunks), "Q&A chunks built")

466 Q&A chunks built


In [27]:
!pip install -q langchain-text-splitters

In [28]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=100)

final_so_chunks = []
for c in so_chunks:
    sub_texts = splitter.split_text(c["text"])
    for sub in sub_texts:
        final_so_chunks.append({"text": sub, "metadata": c["metadata"]})

print(len(final_so_chunks), "final SO chunks after splitting")

1326 final SO chunks after splitting


In [29]:
texts = [c["text"] for c in final_so_chunks]
metadatas = [c["metadata"] for c in final_so_chunks]
ids = [f"so_{c['metadata']['question_id']}_{i}" for i, c in enumerate(final_so_chunks)]

embeddings = model.encode(texts, show_progress_bar=True, batch_size=64)

collection.add(
    embeddings=embeddings.tolist(),
    documents=texts,
    metadatas=metadatas,
    ids=ids
)

print(collection.count(), "total chunks in collection now")

Batches:   0%|          | 0/21 [00:00<?, ?it/s]

2994 total chunks in collection now


In [30]:

from google.colab import userdata

# Ensure these secrets are set in Colab's secret manager
Chroma_Api_Key = userdata.get('Chroma_Api_Key')
Chroma_Tenant = userdata.get('Chroma_Tenant')
Chroma_Database = userdata.get('Chroma_Database')

client = chromadb.CloudClient(
  api_key=Chroma_Api_Key ,
  tenant=Chroma_Tenant,
  database=Chroma_Database
)

In [46]:
oversized = [(i, len(c["text"].encode('utf-8'))) for i, c in enumerate(all_source_chunks) if len(c["text"].encode('utf-8')) > 16384]
print(len(oversized), "chunks exceed the 16KB limit")
print(oversized[:10])

5 chunks exceed the 16KB limit
[(27, 38369), (266, 21641), (293, 20165), (657, 16772), (994, 22702)]


In [47]:
print("Re-chunking oversized documents...")

oversized_indices = {idx for idx, _ in oversized}
rechunked_source_chunks = []

for i, chunk in enumerate(all_source_chunks):
    if i in oversized_indices:
        # Split the oversized chunk
        sub_texts = splitter.split_text(chunk["text"])
        for sub_text in sub_texts:
            rechunked_source_chunks.append({"text": sub_text, "metadata": chunk["metadata"]})
    else:
        # Keep the chunk as is if not oversized
        rechunked_source_chunks.append(chunk)

all_source_chunks = rechunked_source_chunks

print(f"Total chunks after re-chunking oversized documents: {len(all_source_chunks)}")

# Verify that there are no more oversized chunks after this process
new_oversized = [(i, len(c["text"].encode('utf-8'))) for i, c in enumerate(all_source_chunks) if len(c["text"].encode('utf-8')) > 16384]
print(f"{len(new_oversized)} chunks still exceed the 16KB limit (expected 0).")

Re-chunking oversized documents...
Total chunks after re-chunking oversized documents: 3188
0 chunks still exceed the 16KB limit (expected 0).


In [48]:
cloud_collection = client.get_or_create_collection(name="git_docs")
print(f"Chroma Cloud collection '{cloud_collection.name}' initialized. Current count: {cloud_collection.count()}")

Chroma Cloud collection 'git_docs' initialized. Current count: 0


In [51]:
# Ensure `splitter` is defined and `all_source_chunks` is correctly prepared
from langchain_text_splitters import RecursiveCharacterTextSplitter
splitter = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=100)

# 1. Combine all source chunks into a single list
# This line ensures all_source_chunks is always built from the latest data
all_source_chunks = all_chunks + all_progit_chunks + final_so_chunks

# 2. Identify oversized chunks within the newly combined list
oversized = [(i, len(c["text"].encode('utf-8'))) for i, c in enumerate(all_source_chunks) if len(c["text"].encode('utf-8')) > 16384]
print(f"Found {len(oversized)} chunks exceeding the 16KB limit before re-chunking.")

# 3. Re-chunk oversized documents
print("Re-chunking oversized documents...")
oversized_indices = {idx for idx, _ in oversized}
rechunked_source_chunks = []

for i, chunk in enumerate(all_source_chunks):
    if i in oversized_indices:
        sub_texts = splitter.split_text(chunk["text"])
        for sub_text in sub_texts:
            rechunked_source_chunks.append({"text": sub_text, "metadata": chunk["metadata"]})
    else:
        rechunked_source_chunks.append(chunk)
all_source_chunks = rechunked_source_chunks

print(f"Total chunks after re-chunking oversized documents: {len(all_source_chunks)}")

# Verify that there are no more oversized chunks after this process
new_oversized = [(i, len(c["text"].encode('utf-8'))) for i, c in enumerate(all_source_chunks) if len(c["text"].encode('utf-8')) > 16384]
print(f"{len(new_oversized)} chunks still exceed the 16KB limit (expected 0).")

# Now, proceed with adding the correctly prepared chunks to Chroma Cloud
texts = [c["text"] for c in all_source_chunks]
metadatas = [c["metadata"] for c in all_source_chunks]
ids = [f"{c['metadata'].get('source','x')}_{i}" for i, c in enumerate(all_source_chunks)]

# Define a batch size, adjusted to your Chroma Cloud quota
batch_size = 300 # Adjusted to respect Chroma Cloud quota

print(f"Starting to add {len(texts)} chunks to Chroma Cloud in batches of {batch_size}...")

for i in range(0, len(texts), batch_size):
    batch_texts = texts[i:i + batch_size]
    batch_metadatas = metadatas[i:i + batch_size]
    batch_ids = ids[i:i + batch_size]

    # Make sure 'model' (SentenceTransformer) is loaded before encoding
    # This assumes 'model' is defined in an earlier cell and available in the kernel
    if 'model' not in locals():
        from sentence_transformers import SentenceTransformer
        model = SentenceTransformer('all-MiniLM-L6-v2')

    embeddings = model.encode(batch_texts, show_progress_bar=True, batch_size=64) # Encode each batch

    cloud_collection.add(
        embeddings=embeddings.tolist(),
        documents=batch_texts,
        metadatas=batch_metadatas,
        ids=batch_ids
    )
    print(f"Added batch {i//batch_size + 1}/{len(texts)//batch_size + 1}. Total chunks in collection: {cloud_collection.count()}")

print(cloud_collection.count(), "chunks pushed to Chroma Cloud")

Found 5 chunks exceeding the 16KB limit before re-chunking.
Re-chunking oversized documents...
Total chunks after re-chunking oversized documents: 3188
0 chunks still exceed the 16KB limit (expected 0).
Starting to add 3188 chunks to Chroma Cloud in batches of 300...


Batches:   0%|          | 0/5 [00:00<?, ?it/s]

Added batch 1/11. Total chunks in collection: 300


Batches:   0%|          | 0/5 [00:00<?, ?it/s]

Added batch 2/11. Total chunks in collection: 600


...

Added batch 9/11. Total chunks in collection: 2700


Batches:   0%|          | 0/5 [00:00<?, ?it/s]

Added batch 10/11. Total chunks in collection: 3000


Batches:   0%|          | 0/3 [00:00<?, ?it/s]

Added batch 11/11. Total chunks in collection: 3188
3188 chunks pushed to Chroma Cloud


In [52]:
# To verify data has been pushed to Chroma Cloud, run a query against the cloud_collection

model = SentenceTransformer('all-MiniLM-L6-v2')
query = "how do I undo my last commit"
query_embedding = model.encode([query])

results = cloud_collection.query(
    query_embeddings=query_embedding.tolist(),
    n_results=5
)

print("\nQuery Results from Chroma Cloud:")
for doc, meta, dist in zip(results['documents'][0], results['metadatas'][0], results['distances'][0]):
    if meta.get('source') == 'progit':
        print(f"[{meta.get('source')} / {meta.get('chapter')} / {meta.get('heading')}] (dist={dist:.3f})")
    elif meta.get('source') == 'stackoverflow':
        print(f"[{meta.get('source')} / {meta.get('title')}] (dist={dist:.3f})")
    else:
        print(f"[{meta.get('command')} / {meta.get('section')}] (dist={dist:.3f})")
    print(doc[:200])
    print()

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]


Query Results from Chroma Cloud:
[stackoverflow / How do I undo the most recent local commits in Git?] (dist=0.522)
git reset
 is the command responsible for the 
undo
. It will undo your last commit while 
leaving your working tree (the state of your files on disk) untouched.
 You'll need to add them again before 

[stackoverflow / How to undo &quot;git commit --amend&quot; done instead of &quot;git commit&quot;] (dist=0.534)
Question: How to undo &quot;git commit --amend&quot; done instead of &quot;git commit&quot;
I accidentally amended my previous commit. The commit should have been separate to keep history of the chang

[stackoverflow / How do I undo the most recent local commits in Git?] (dist=0.567)
Question: How do I undo the most recent local commits in Git?
I accidentally committed the wrong files to 
Git
 but haven't pushed the commit to the server yet.


How do I undo those commits from the 

[stackoverflow / How do I undo &#39;git add&#39; before commit?] (dist=0.590)
Que

In [53]:
print(f"\nChecking Chroma Cloud collection count: {cloud_collection.count()} chunks.")

# Re-run the query to verify data presence
model = SentenceTransformer('all-MiniLM-L6-v2')
query = "how do I undo my last commit"
query_embedding = model.encode([query])

results = cloud_collection.query(
    query_embeddings=query_embedding.tolist(),
    n_results=5
)

print("\nQuery Results from Chroma Cloud (Verification):")
if results['documents'] and len(results['documents'][0]) > 0:
    for doc, meta, dist in zip(results['documents'][0], results['metadatas'][0], results['distances'][0]):
        if meta.get('source') == 'progit':
            print(f"[{meta.get('source')} / {meta.get('chapter')} / {meta.get('heading')}] (dist={dist:.3f})")
        elif meta.get('source') == 'stackoverflow':
            print(f"[{meta.get('source')} / {meta.get('title')}] (dist={dist:.3f})")
        else:
            print(f"[{meta.get('command')} / {meta.get('section')}] (dist={dist:.3f})")
        print(doc[:200])
        print()
else:
    print("No results found. This might indicate an issue with data ingestion or the query itself.")


Checking Chroma Cloud collection count: 3188 chunks.


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]


Query Results from Chroma Cloud (Verification):
[stackoverflow / How do I undo the most recent local commits in Git?] (dist=0.522)
git reset
 is the command responsible for the 
undo
. It will undo your last commit while 
leaving your working tree (the state of your files on disk) untouched.
 You'll need to add them again before 

[stackoverflow / How to undo &quot;git commit --amend&quot; done instead of &quot;git commit&quot;] (dist=0.534)
Question: How to undo &quot;git commit --amend&quot; done instead of &quot;git commit&quot;
I accidentally amended my previous commit. The commit should have been separate to keep history of the chang

[stackoverflow / How do I undo the most recent local commits in Git?] (dist=0.567)
Question: How do I undo the most recent local commits in Git?
I accidentally committed the wrong files to 
Git
 but haven't pushed the commit to the server yet.


How do I undo those commits from the 

[stackoverflow / How do I undo &#39;git add&#39; before commit?] (

In [55]:
print(f"\nChecking Chroma Cloud collection count: {cloud_collection.count()} chunks.")

# Re-run the query to verify data presence
model = SentenceTransformer('all-MiniLM-L6-v2')
query = "how do I undo my last commit"
query_embedding = model.encode([query])

results = cloud_collection.query(
    query_embeddings=query_embedding.tolist(),
    n_results=5
)

print("\nQuery Results from Chroma Cloud (Verification):")
if results['documents'] and len(results['documents'][0]) > 0:
    for doc, meta, dist in zip(results['documents'][0], results['metadatas'][0], results['distances'][0]):
        if meta.get('source') == 'progit':
            print(f"[{meta.get('source')} / {meta.get('chapter')} / {meta.get('heading')}] (dist={dist:.3f})")
        elif meta.get('source') == 'stackoverflow':
            print(f"[{meta.get('source')} / {meta.get('title')}] (dist={dist:.3f})")
        else:
            print(f"[{meta.get('command')} / {meta.get('section')}] (dist={dist:.3f})")
        print(doc[:200])
        print()
else:
    print("No results found. This might indicate an issue with data ingestion or the query itself.")


Checking Chroma Cloud collection count: 3188 chunks.


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]


Query Results from Chroma Cloud (Verification):
[stackoverflow / How do I undo the most recent local commits in Git?] (dist=0.522)
git reset
 is the command responsible for the 
undo
. It will undo your last commit while 
leaving your working tree (the state of your files on disk) untouched.
 You'll need to add them again before 

[stackoverflow / How to undo &quot;git commit --amend&quot; done instead of &quot;git commit&quot;] (dist=0.534)
Question: How to undo &quot;git commit --amend&quot; done instead of &quot;git commit&quot;
I accidentally amended my previous commit. The commit should have been separate to keep history of the chang

[stackoverflow / How do I undo the most recent local commits in Git?] (dist=0.567)
Question: How do I undo the most recent local commits in Git?
I accidentally committed the wrong files to 
Git
 but haven't pushed the commit to the server yet.


How do I undo those commits from the 

[stackoverflow / How do I undo &#39;git add&#39; before commit?] (

In [2]:
!pip install -q sentence-transformers chromadb

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 75.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 27.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 131.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.2/19.2 MB 97.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.2/137.2 kB 13.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.6/204.6 kB 20.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 6.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently

In [2]:
import chromadb
from sentence_transformers import SentenceTransformer
from google.colab import userdata

Chroma_Api_Key = userdata.get('Chroma_Api_Key')
Chroma_Tenant = userdata.get('Chroma_Tenant')
Chroma_Database = userdata.get('Chroma_Database')

client = chromadb.CloudClient(
    api_key=Chroma_Api_Key,
    tenant=Chroma_Tenant,
    database=Chroma_Database
)
collection = client.get_or_create_collection(name="git_docs")
print(collection.count(), "chunks found")  # should match your pushed count, e.g. ~2000+

model = SentenceTransformer('all-MiniLM-L6-v2')
query_embedding = model.encode(["how do I undo my last commit"])
results = collection.query(query_embeddings=query_embedding.tolist(), n_results=3)
for doc, meta in zip(results['documents'][0], results['metadatas'][0]):
    print(meta.get('source'), "-", doc[:100])

3188 chunks found


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

stackoverflow - git reset
 is the command responsible for the 
undo
. It will undo your last commit while 
leaving y
stackoverflow - Question: How to undo &quot;git commit --amend&quot; done instead of &quot;git commit&quot;
I accide
stackoverflow - Question: How do I undo the most recent local commits in Git?
I accidentally committed the wrong fil


In [5]:
import chromadb
print(chromadb.__version__)

from chromadb import Schema, SparseVectorIndexConfig, Search, Knn, Rrf, K
from chromadb.utils.embedding_functions import ChromaBm25EmbeddingFunction
from google.colab import userdata

Chroma_Api_Key = userdata.get('Chroma_Api_Key')
Chroma_Tenant = userdata.get('Chroma_Tenant')
Chroma_Database = userdata.get('Chroma_Database')

client = chromadb.CloudClient(
    api_key=Chroma_Api_Key,
    tenant=Chroma_Tenant,
    database=Chroma_Database
)

bm25_ef = ChromaBm25EmbeddingFunction(
    k=1.2,
    b=0.75,
    avg_doc_length=256.0,
    token_max_length=40
)

schema = Schema().create_index(
    key='sparse_vector_key',
    config=SparseVectorIndexConfig(
        embedding_function=bm25_ef,
        source_key=K.DOCUMENT,
        bm25=True
    )
)

cloud_collection = client.get_or_create_collection(name="git_docs", schema=schema)
print(cloud_collection.count(), "chunks confirmed in cloud")

1.5.9
3188 chunks confirmed in cloud


In [6]:
from chromadb import Search, Knn, Rrf

def hybrid_retrieve(query: str, k: int = 20, candidate_pool: int = 200, dense_weight=0.6, sparse_weight=0.4):
    dense_rank = Knn(query=query, return_rank=True, limit=candidate_pool)
    sparse_rank = Knn(query=query, key="sparse_vector_key", return_rank=True, limit=candidate_pool)

    hybrid_rank = Rrf(
        ranks=[dense_rank, sparse_rank],
        weights=[dense_weight, sparse_weight],
        k=60
    )

    search = (
        Search()
        .rank(hybrid_rank)
        .limit(k)
        .select(K.DOCUMENT, K.SCORE, "source", "section", "command", "chapter", "heading", "title")
    )

    results = cloud_collection.search(search)
    return results

# quick test
r = hybrid_retrieve("how do I undo my last commit")
print(r)

/root/.cache/chroma/onnx_models/all-MiniLM-L6-v2/onnx.tar.gz: 100%|██████████| 79.3M/79.3M [00:02<00:00, 32.0MiB/s]


ValueError: Cannot embed string query for key 'sparse_vector_key': key not found in schema. Please provide an embedded vector or configure an embedding function for this key in the schema.